# RAG Assignment: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** *29 CFR 1910 – General Industry Special Requirements (2026)*, a 378-page OSHA regulatory guide.

**Pipeline 1 (baseline):** bi-encoder → FAISS top-10 → first 3 chunks → LLM → Answer 1  
**Pipeline 2 (re-ranking):** same top-10 → cross-encoder scores every (query, chunk) pair → re-sorted top 3 → LLM (same prompt) → Answer 2

## Step 1: Locate the PDF

In [1]:
from pathlib import Path

DATA_DIR = Path("data")
pdf_files = list(DATA_DIR.glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError(
        "No PDF found. Put your PDF inside the data/ folder."
    )

if len(pdf_files) > 1:
    raise ValueError(
        "Multiple PDFs found. Keep only your chosen assignment PDF in data/."
    )

PDF_PATH = pdf_files[0]

print("PDF selected:", PDF_PATH)
print("File size (MB):", round(PDF_PATH.stat().st_size / (1024 * 1024), 2))

PDF selected: data/29 CFR 1910 - General Industry Special Requirements 2026.pdf
File size (MB): 3.29


## Step 2: Extract text page by page (PyMuPDF)

In [2]:
import pymupdf

pages = []

with pymupdf.open(PDF_PATH) as doc:
    print("Total PDF pages:", len(doc))

    for page_index, page in enumerate(doc):
        text = page.get_text("text", sort=True).strip()

        pages.append({
            "page_number": page_index + 1,
            "text": text
        })

print("Pages processed:", len(pages))
print("Pages containing text:", sum(bool(p["text"]) for p in pages))

Total PDF pages: 378


Pages processed: 378
Pages containing text: 378


In [3]:
for page in pages[:3]:
    print(f"\n--- PDF Page {page['page_number']} ---")
    print(page["text"][:1200])


--- PDF Page 1 ---
29 CFR 1910 – General Industry

Note: NC State-Speciﬁc Standards that are relevant to General Industry are included at the end.
       OSHA Standards With Special Requirements

These webpages are designed to assist employers in complying with standards that
have "special requirements". For the purpose of this technical guidance, special
requirements includes programs, plans, procedures, policies, inspections, certiﬁcations,
competent persons, qualiﬁed persons, training, medical surveillance, exposure
monitoring, signs, tags, markings, tests, records and recordkeeping requirements that are
applicable in meeting the requirements of OSHA standards by industry. If a standard or
subpart are not listed, it did not have one of these types of special requirements.

Note: While all attempts have been made to ensure the accuracy of the content and
completeness in the information provided, it may not include all the speciﬁc standards that
have special requirements that may be 

## Step 3: Chunk the text
Fixed-size character windows with overlap, built per page so that each chunk keeps its page number.

In [4]:
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 300

assert CHUNK_SIZE > CHUNK_OVERLAP > 0

chunks = []

for page in pages:
    text = page["text"]

    # Skip pages that contain no extractable text.
    if not text:
        continue

    start = 0

    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "chunk_id": f"C{len(chunks) + 1:04d}",
                "page_number": page["page_number"],
                "text": chunk_text
            })

        if end == len(text):
            break

        start = end - CHUNK_OVERLAP

print("Total chunks created:", len(chunks))

Total chunks created: 757


In [5]:
import pandas as pd

chunks_df = pd.DataFrame(chunks)

display(chunks_df.head(5))

print("Number of chunks:", len(chunks_df))
print("Unique chunk IDs:", chunks_df["chunk_id"].nunique())
print("Missing text values:", chunks_df["text"].isna().sum())

,chunk_id,page_number,text
0,C0001,1,29 CFR 1910 – General Industry\n\nNote: NC Sta...
1,C0002,1,"ed Platforms, Manlifts and Vehicle Mounted Wor..."
2,C0003,2,• Subpart M - Compressed Gas and Compressed A...
3,C0004,3,1910.22(a)(3) - Walking-working surfaces are m...
4,C0005,3,"ders, mobile ladder stands and platforms. This..."


Number of chunks: 757
Unique chunk IDs: 757
Missing text values: 0


In [6]:
import json

OUTPUT_PATH = Path("data/chunks.jsonl")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("Saved chunks to:", OUTPUT_PATH)

Saved chunks to: data/chunks.jsonl


In [7]:
assert len(chunks) > 0, "No chunks were created."
assert chunks_df["chunk_id"].is_unique, "Chunk IDs are not unique."
assert chunks_df["text"].str.strip().ne("").all(), "Empty chunk found."

print("Step 3 completed successfully!")
print(f"Pages extracted: {len(pages)}")
print(f"Chunks created: {len(chunks)}")
print(f"Average chunk length: {chunks_df['text'].str.len().mean():.0f} characters")

Step 3 completed successfully!
Pages extracted: 378
Chunks created: 757
Average chunk length: 1460 characters


# Step 4 — Generate embeddings and build the FAISS index

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Embedding dimensions:", embedding_model.get_sentence_embedding_dimension())

W1019 10:22:29.599000 33385 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimensions: 384


/var/folders/nf/ywxk5fv95hn3pqx22yp_c68c0000gn/T/ipykernel_33385/8903787.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", embedding_model.get_sentence_embedding_dimension())


In [9]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print("Embedding matrix shape:", embeddings.shape)
print("Number of chunks:", len(chunk_texts))

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedding matrix shape: (757, 384)
Number of chunks: 757


In [10]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created.")
print("Vectors stored:", index.ntotal)

FAISS index created.
Vectors stored: 757


### Step 4b: How much of each chunk does the bi-encoder actually see?
`all-MiniLM-L6-v2` truncates input at 256 word-piece tokens, while the cross-encoder reads up to 512 tokens for the (query + chunk) pair.
Any clause near the end of a long chunk is invisible to the bi-encoder embedding. This turns out to matter for several failure cases below.

In [11]:
# Count tokens per chunk with the bi-encoder's own tokenizer
token_counts = [len(embedding_model.tokenizer(c["text"])["input_ids"]) for c in chunks]
bi_max_len = embedding_model.max_seq_length

print("Bi-encoder max_seq_length:", bi_max_len)
print(f"Median tokens per chunk: {int(np.median(token_counts))}")
print(f"Chunks truncated by the bi-encoder: {sum(t > bi_max_len for t in token_counts)} / {len(chunks)} "
      f"({100 * np.mean([t > bi_max_len for t in token_counts]):.1f}%)")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (354 > 256). Running this sequence through the model will result in indexing errors


Bi-encoder max_seq_length: 256
Median tokens per chunk: 352
Chunks truncated by the bi-encoder: 569 / 757 (75.2%)


## Step 5: Pipeline 1, baseline retrieval (bi-encoder)
The query is embedded with the **same** bi-encoder, and FAISS returns the 10 nearest chunks.
Because the embeddings are L2-normalised, the inner product (`IndexFlatIP`) equals cosine similarity.
Only the **first 3** of those 10 are given to the LLM.

In [12]:
import time

TOP_K_RETRIEVE = 10   # candidates pulled from the vector store
TOP_K_CONTEXT = 3     # chunks that actually go into the LLM prompt


def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    """Return the top-k chunks for `query`, ranked by cosine similarity (bi-encoder)."""
    query_vec = embedding_model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_vec, k)

    results = []
    for rank, (chunk_idx, score) in enumerate(zip(ids[0], scores[0]), start=1):
        results.append({"bi_rank": rank, "bi_score": float(score), **chunks[chunk_idx]})
    return results


# Quick check
for r in baseline_retrieve("How long must employee exposure records be retained?")[:3]:
    print(r["bi_rank"], r["chunk_id"], f"p{r['page_number']}", round(r["bi_score"], 3), r["text"][:90].replace("\n", " "))

1 C0537 p270 0.725 eserved and maintained for at least the duration of       employment plus thirty (30) year
2 C0065 p33 0.71 test rooms.  1910.95(m)(3) - Record retention. The employer shall retain records required 
3 C0653 p327 0.643 y paragraph (e) of this               section, in accordance with 29 CFR 1910.1020 - acces


## Step 6: Pipeline 2, re-ranking with a cross-encoder
All 10 baseline candidates are scored **jointly** with the query by `cross-encoder/ms-marco-MiniLM-L-6-v2`
(the Hugging Face model id for the "msmarco-MiniLM-L-6-v2" cross-encoder named in the brief).
Candidates are re-sorted by that score, and the new top 3 go to the LLM.

In [13]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)
print("Cross-encoder loaded:", RERANKER_MODEL_NAME, "| max_length:", reranker.max_length)


def rerank(query, candidates):
    """Score every (query, chunk) pair with the cross-encoder and sort by that score, highest first."""
    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = reranker.predict(pairs)

    reranked = [{**c, "ce_score": float(s)} for c, s in zip(candidates, ce_scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)

    for new_rank, c in enumerate(reranked, start=1):
        c["ce_rank"] = new_rank
    return reranked

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2 | max_length: 512


/var/folders/nf/ywxk5fv95hn3pqx22yp_c68c0000gn/T/ipykernel_33385/2297532135.py:5: DeprecationWarning: The `max_length` property was renamed and is now deprecated. Please use `max_seq_length` instead.
  print("Cross-encoder loaded:", RERANKER_MODEL_NAME, "| max_length:", reranker.max_length)


## Step 7: LLM answer generation (identical prompt for both pipelines)
Uses the Gemini API (`google-genai`). Put `GEMINI_API_KEY=...` in `.env`.
If no key is found, the retrieval experiments still run and the answers are skipped.
`temperature=0` keeps the comparison deterministic, so any difference between Answer 1 and Answer 2 comes from the context.

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

llm_client = None
if api_key:
    from google import genai
    from google.genai import types
    llm_client = genai.Client(api_key=api_key)
    print("Gemini client ready, model:", GEMINI_MODEL)
else:
    print("No GEMINI_API_KEY found in .env. LLM answers will be skipped.")


PROMPT_TEMPLATE = """You are an assistant that answers questions about OSHA 29 CFR 1910 (General Industry) requirements.
Answer the question using ONLY the context below. Quote the regulation numbers (e.g. 1910.147(e)(3)) that support your answer.
If the context does not contain the answer, reply exactly: "The provided context does not answer this."

Context:
{context}

Question: {question}

Answer:"""


def build_context(selected_chunks):
    return "\n\n".join(
        f"[Chunk {c['chunk_id']} | page {c['page_number']}]\n{c['text']}" for c in selected_chunks
    )


def generate_answer(question, selected_chunks, max_retries=8):
    """Send the same prompt template to the LLM; return (answer_text, seconds of the successful call).
    The free Gemini tier allows ~5 requests/min, so we pace calls and retry on HTTP 429/503."""
    if llm_client is None:
        return "(skipped: no API key)", 0.0

    prompt = PROMPT_TEMPLATE.format(context=build_context(selected_chunks), question=question)
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            response = llm_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0),
            )
            elapsed = time.perf_counter() - t0
            time.sleep(13)   # stay under the free-tier rate limit
            return response.text.strip(), elapsed
        except Exception as e:
            if any(code in str(e) for code in ("429", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE")):
                time.sleep(40)   # wait out the per-minute quota window
            else:
                raise
    return "(failed: rate limit)", 0.0


Gemini client ready, model: gemini-3.5-flash-lite


## Step 8: Run both pipelines end to end and show the rank shifts

In [15]:
from IPython.display import display, Markdown


def run_both_pipelines(query, with_llm=True):
    """Run Pipeline 1 and Pipeline 2 on one query, timing each stage."""
    t0 = time.perf_counter()
    candidates = baseline_retrieve(query)            # Pipeline 1 (a, b)
    t_retrieve = time.perf_counter() - t0

    t0 = time.perf_counter()
    reranked = rerank(query, candidates)             # Pipeline 2 (a, b, c)
    t_rerank = time.perf_counter() - t0

    top3_baseline = candidates[:TOP_K_CONTEXT]       # Pipeline 1 (c)
    top3_reranked = reranked[:TOP_K_CONTEXT]         # Pipeline 2 (c)

    answer1, t_llm1 = generate_answer(query, top3_baseline) if with_llm else ("", 0.0)
    answer2, t_llm2 = generate_answer(query, top3_reranked) if with_llm else ("", 0.0)

    return {
        "query": query, "candidates": candidates, "reranked": reranked,
        "top3_baseline": top3_baseline, "top3_reranked": top3_reranked,
        "answer1": answer1, "answer2": answer2,
        "t_retrieve": t_retrieve, "t_rerank": t_rerank, "t_llm1": t_llm1, "t_llm2": t_llm2,
    }


def rank_shift_table(result, gold_ids=()):
    """One row per candidate: bi-encoder rank vs cross-encoder rank (gold chunks marked with ★)."""
    rows = []
    for c in result["reranked"]:
        rows.append({
            "chunk": ("★ " if c["chunk_id"] in gold_ids else "") + c["chunk_id"],
            "page": c["page_number"],
            "bi_rank": c["bi_rank"],
            "ce_rank": c["ce_rank"],
            "shift": f"#{c['bi_rank']} → #{c['ce_rank']}",
            "bi_score": round(c["bi_score"], 3),
            "ce_score": round(c["ce_score"], 2),
            "in_baseline_top3": c["bi_rank"] <= TOP_K_CONTEXT,
            "in_rerank_top3": c["ce_rank"] <= TOP_K_CONTEXT,
            "preview": " ".join(c["text"].split())[:110],
        })
    return pd.DataFrame(rows).sort_values("bi_rank").reset_index(drop=True)

## Step 9: Failure mode analysis (stress tests)
Each test query has a **gold chunk** that I identified by reading the regulation text in the PDF.
These are the chunks that actually contain the governing rule, exception or prohibition.
The categories follow the brief: exceptions, negation/constraints, keyword traps, and buried chunks (rank 4–10 → top 3).

The last two queries are **counter-examples**: keyword traps where the cross-encoder itself is distracted. I kept them in to show the limits of re-ranking honestly.

In [16]:
FAILURE_QUERIES = [
    {
        "category": "Exception / corner case",
        "query": "Is fall protection needed while assessing workplace conditions before work begins?",
        "gold": ["C0014"],
        "gold_rule": "1910.28(a)(2)(ii): the duty to have fall protection does NOT apply when inspecting, "
                     "investigating or assessing workplace conditions before work starts or after it is complete.",
    },
    {
        "category": "Exception / buried chunk",
        "query": "Who can serve as the attendant for a confined space entry and can they monitor more than one space?",
        "gold": ["C0412"],
        "gold_rule": "1910.269(e)(7): the attendant is not precluded from performing other duties outside the "
                     "enclosed space, provided they do not distract from monitoring the entrants.",
    },
    {
        "category": "Negation & constraints",
        "query": "Can an employee with a beard wear a tight-fitting respirator?",
        "gold": ["C0160", "C0162"],
        "gold_rule": "1910.134(f): tight-fitting facepieces must pass a fit test; 1910.134(g): the employer must "
                     "prohibit conditions that may result in facepiece seal leakage.",
    },
    {
        "category": "Exception / role limit",
        "query": "Who may remove another worker's lockout device?",
        "gold": ["C0209"],
        "gold_rule": "1910.147(e)(3): each lock is removed by the employee who applied it; exception: removal under the "
                     "employer's direction using a documented, trained procedure.",
    },
    {
        "category": "Keyword trap (counter-example)",
        "query": "Does a small company need a written fire prevention plan?",
        "gold": ["C0034"],
        "gold_rule": "1910.39(b): fire prevention plan must be written, except employers with 10 or fewer employees "
                     "may communicate it orally.",
    },
    {
        "category": "Keyword trap (counter-example)",
        "query": "How often must fire extinguishers be hydrostatically tested?",
        "gold": ["C0224"],
        "gold_rule": "1910.157(e)-(f): hydrostatic testing by trained persons at the intervals in Table L-1; "
                     "stored-pressure dry chemical units emptied and maintained every 6 years.",
    },
]

all_results = []
for fq in FAILURE_QUERIES:
    res = run_both_pipelines(fq["query"])
    res.update(fq)
    all_results.append(res)

    display(Markdown(f"### [{fq['category']}] {fq['query']}\n**Gold clause:** {fq['gold_rule']}"))
    display(rank_shift_table(res, fq["gold"]))
    print("Baseline top-3 :", [c["chunk_id"] for c in res["top3_baseline"]])
    print("Re-ranked top-3:", [c["chunk_id"] for c in res["top3_reranked"]])
    display(Markdown(f"**Answer 1 (baseline):**\n\n{res['answer1']}\n\n**Answer 2 (re-ranked):**\n\n{res['answer2']}\n\n---"))

### [Exception / corner case] Is fall protection needed while assessing workplace conditions before work begins?
**Gold clause:** 1910.28(a)(2)(ii): the duty to have fall protection does NOT apply when inspecting, investigating or assessing workplace conditions before work starts or after it is complete.

,chunk,page,bi_rank,ce_rank,shift,bi_score,ce_score,in_baseline_top3,in_rerank_top3,preview
0,C0178,90,1,7,#1 → #7,0.647,0.16,True,False,"1910.140(c)(13)(ii) - Designed, installed, and..."
1,C0740,369,2,6,#2 → #6,0.628,0.93,True,False,utilized in accordance with the manufacturer's...
2,C0737,368,3,1,#3 → #1,0.624,4.24,True,True,(a) In addition to the inspections required by...
3,★ C0014,8,4,2,#4 → #2,0.616,3.95,False,True,"Qualiﬁed describes a person who, by possession..."
4,C0414,208,5,5,#5 → #5,0.602,1.89,False,False,by paragraph (g)(2)(iii)(L) of this section af...
5,C0015,8,6,4,#6 → #4,0.598,2.58,False,False,ion does not apply when fall protection system...
6,C0742,370,7,3,#7 → #3,0.596,3.43,False,True,"hazard, and the work can not be completed util..."
7,C0021,11,8,8,#8 → #8,0.584,-0.15,False,False,l components) an employer uses to provide prot...
8,C0738,368,9,9,#9 → #9,0.583,-1.21,False,False,s. Special Requirements: Fall protection syste...
9,C0025,13,10,10,#10 → #10,0.575,-2.13,False,False,ciﬁed elsewhere in this subpart. Employers mus...


Baseline top-3 : ['C0178', 'C0740', 'C0737']
Re-ranked top-3: ['C0737', 'C0014', 'C0742']


**Answer 1 (baseline):**

The provided context does not answer this.

**Answer 2 (re-ranked):**

Based on the provided context, fall protection is not required when employers are inspecting, investigating, or assessing workplace conditions or work to be performed prior to the start of work, unless fall protection systems or equipment meeting the requirements of 1910.29 have been installed and are available for workers to use.

Regulation supporting this answer: **1910.28(a)(2)(ii)**

---

### [Exception / buried chunk] Who can serve as the attendant for a confined space entry and can they monitor more than one space?
**Gold clause:** 1910.269(e)(7): the attendant is not precluded from performing other duties outside the enclosed space, provided they do not distract from monitoring the entrants.

...

**Answer 1 (baseline):**

Based on the provided context, a fire prevention plan must be in writing, kept in the workplace, and made available to employees for review, except that an employer with 10 or fewer employees may communicate the plan orally to employees (1910.39(b)).

**Answer 2 (re-ranked):**

Based on the provided context, an employer with 10 or fewer employees is permitted to communicate the fire prevention plan orally rather than having it in writing (1910.39(b)). 

Supporting regulation: 1910.39(b)

---

### [Keyword trap (counter-example)] How often must fire extinguishers be hydrostatically tested?
**Gold clause:** 1910.157(e)-(f): hydrostatic testing by trained persons at the intervals in Table L-1; stored-pressure dry chemical units emptied and maintained every 6 years.

,chunk,page,bi_rank,ce_rank,shift,bi_score,ce_score,in_baseline_top3,in_rerank_top3,preview
0,★ C0224,113,1,1,#1 → #1,0.676,1.38,True,True,1910.157(e)(1) - The employer shall be respons...
1,C0234,118,2,4,#2 → #4,0.518,-5.31,True,False,employee alarm shall provide employees time to...
2,C0225,113,3,6,#3 → #6,0.502,-7.35,True,False,yees with the general principles of ﬁre exting...
3,C0222,112,4,3,#4 → #3,0.496,-2.33,False,True,1910.157 - Portable Fire Extinguishers Scope: ...
4,C0232,117,5,7,#5 → #7,0.494,-8.00,False,False,"1910.161 - Fixed Extinguishing Systems, Dry Ch..."
5,C0231,116,6,10,#6 → #10,0.461,-10.47,False,False,sure that the weight and pressure of reﬁllable...
6,C0080,41,7,5,#7 → #5,0.455,-5.74,False,False,"Special Requirements: Designs, inspections, ma..."
7,C0076,39,8,2,#8 → #2,0.453,0.61,False,True,"paragraph, shall be considered prima facie evi..."
8,C0487,244,9,8,#9 → #8,0.451,-8.51,False,False,"and 1926.912 - underwater blasting, of Title 2..."
9,C0088,45,10,9,#10 → #9,0.449,-9.57,False,False,"primers, smokeless propellant, cartridges for ..."


Baseline top-3 : ['C0224', 'C0234', 'C0225']
Re-ranked top-3: ['C0224', 'C0076', 'C0222']


**Answer 1 (baseline):**

The provided context does not answer this.

**Answer 2 (re-ranked):**

The provided context does not answer this.

---

### Step 9b: Summary, where the gold chunk landed in each pipeline

In [17]:
def best_gold_rank(ranked, gold, key):
    ranks = [c[key] for c in ranked if c["chunk_id"] in gold]
    return min(ranks) if ranks else None   # None = gold not in the top 10 at all

summary_rows = []
for r in all_results:
    b = best_gold_rank(r["candidates"], r["gold"], "bi_rank")
    c = best_gold_rank(r["reranked"], r["gold"], "ce_rank")
    summary_rows.append({
        "category": r["category"],
        "query": r["query"],
        "gold": ", ".join(r["gold"]),
        "gold_rank_baseline": b,
        "gold_rank_reranked": c,
        "gold_in_top3_baseline": b is not None and b <= TOP_K_CONTEXT,
        "gold_in_top3_reranked": c is not None and c <= TOP_K_CONTEXT,
        "baseline_top3": ", ".join(x["chunk_id"] for x in r["top3_baseline"]),
        "reranked_top3": ", ".join(x["chunk_id"] for x in r["top3_reranked"]),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

print("Gold hit@3, baseline :", summary_df["gold_in_top3_baseline"].sum(), "/", len(summary_df))
print("Gold hit@3, reranked :", summary_df["gold_in_top3_reranked"].sum(), "/", len(summary_df))
print("MRR@10 baseline:", round(np.mean([1 / x for x in summary_df["gold_rank_baseline"] if x]), 3))
print("MRR@10 reranked:", round(np.mean([1 / x for x in summary_df["gold_rank_reranked"] if x]), 3))

,category,query,gold,gold_rank_baseline,gold_rank_reranked,gold_in_top3_baseline,gold_in_top3_reranked,baseline_top3,reranked_top3
0,Exception / corner case,Is fall protection needed while assessing work...,C0014,4,2,False,True,"C0178, C0740, C0737","C0737, C0014, C0742"
1,Exception / buried chunk,Who can serve as the attendant for a confined ...,C0412,4,1,False,True,"C0201, C0445, C0411","C0412, C0411, C0410"
2,Negation & constraints,Can an employee with a beard wear a tight-fitt...,"C0160, C0162",6,1,False,True,"C0153, C0159, C0588","C0160, C0159, C0162"
3,Exception / role limit,Who may remove another worker's lockout device?,C0209,2,1,True,True,"C0211, C0209, C0407","C0209, C0410, C0407"
4,Keyword trap (counter-example),Does a small company need a written fire preve...,C0034,1,2,True,True,"C0034, C0433, C0222","C0645, C0034, C0714"
5,Keyword trap (counter-example),How often must fire extinguishers be hydrostat...,C0224,1,1,True,True,"C0224, C0234, C0225","C0224, C0076, C0222"


Gold hit@3, baseline : 3 / 6
Gold hit@3, reranked : 6 / 6
MRR@10 baseline: 0.528
MRR@10 reranked: 0.833


## Step 10: Latency profiling
Each stage is timed over repeated runs (after a warm-up) so the numbers are stable.
- **Baseline retrieval** = embed query + FAISS search over all chunks
- **Re-ranking** = cross-encoder forward pass over the 10 candidates
- **LLM** = one Gemini call (network-bound; measured once per answer in Step 9)

In [18]:
N_REPEATS = 20
queries = [fq["query"] for fq in FAILURE_QUERIES]

# Warm-up (first calls include lazy initialisation)
for q in queries:
    rerank(q, baseline_retrieve(q))

latency_rows = []
for q in queries:
    bi_times, ce_times = [], []
    for _ in range(N_REPEATS):
        t0 = time.perf_counter(); cands = baseline_retrieve(q); bi_times.append(time.perf_counter() - t0)
        t0 = time.perf_counter(); rerank(q, cands);              ce_times.append(time.perf_counter() - t0)
    latency_rows.append({"query": q[:60], "baseline_ms": 1000 * np.median(bi_times), "rerank_ms": 1000 * np.median(ce_times)})

latency_df = pd.DataFrame(latency_rows)
latency_df["pipeline2_total_ms"] = latency_df["baseline_ms"] + latency_df["rerank_ms"]
latency_df["overhead_x"] = latency_df["pipeline2_total_ms"] / latency_df["baseline_ms"]
display(latency_df.round(2))

llm_times = [t for r in all_results for t in (r["t_llm1"], r["t_llm2"]) if t > 0]

latency_summary = pd.DataFrame([
    {"stage": "Pipeline 1: retrieval (embed + FAISS top-10)", "median_ms": latency_df["baseline_ms"].median()},
    {"stage": "Pipeline 2: cross-encoder re-rank of 10",      "median_ms": latency_df["rerank_ms"].median()},
    {"stage": "Pipeline 2: total retrieval",                  "median_ms": latency_df["pipeline2_total_ms"].median()},
    {"stage": "LLM generation (per answer)",                  "median_ms": 1000 * np.median(llm_times) if llm_times else np.nan},
])
display(latency_summary.round(2))

,query,baseline_ms,rerank_ms,pipeline2_total_ms,overhead_x
0,Is fall protection needed while assessing work...,5.11,73.04,78.15,15.31
1,Who can serve as the attendant for a confined ...,4.04,79.88,83.92,20.77
2,Can an employee with a beard wear a tight-fitt...,4.58,79.68,84.25,18.41
3,Who may remove another worker's lockout device?,5.27,72.26,77.53,14.71
4,Does a small company need a written fire preve...,6.49,74.66,81.15,12.50
5,How often must fire extinguishers be hydrostat...,4.70,72.08,76.78,16.34


,stage,median_ms
0,Pipeline 1: retrieval (embed + FAISS top-10),4.90
1,Pipeline 2: cross-encoder re-rank of 10,73.85
2,Pipeline 2: total retrieval,79.65
3,LLM generation (per answer),1409.65


## Step 11: Scaling experiment (why not cross-encode the whole collection?)
For one query, compare the cost of scoring **every** chunk with each model.
The bi-encoder's chunk embeddings were computed once, offline, so at query time only the query is encoded.
A cross-encoder has nothing to pre-compute, because every (query, chunk) pair needs a fresh forward pass.

In [19]:
q = FAILURE_QUERIES[0]["query"]
all_texts = [c["text"] for c in chunks]

t0 = time.perf_counter()
qv = embedding_model.encode([q], normalize_embeddings=True).astype("float32")
index.search(qv, len(chunks))
t_bi_all = time.perf_counter() - t0

t0 = time.perf_counter()
reranker.predict([(q, t) for t in all_texts], batch_size=32)
t_ce_all = time.perf_counter() - t0

per_pair = t_ce_all / len(chunks)
scaling_df = pd.DataFrame([
    {"approach": f"Bi-encoder over all {len(chunks)} chunks", "seconds": t_bi_all},
    {"approach": f"Cross-encoder over all {len(chunks)} chunks", "seconds": t_ce_all},
    {"approach": "Cross-encoder over top-10 only", "seconds": per_pair * 10},
    {"approach": "Cross-encoder over 1,000,000 chunks (extrapolated)", "seconds": per_pair * 1_000_000},
])
display(scaling_df.round(4))
print(f"Cross-encoder cost ≈ {1000 * per_pair:.2f} ms per (query, chunk) pair on this machine")

,approach,seconds
0,Bi-encoder over all 757 chunks,0.0070
1,Cross-encoder over all 757 chunks,4.6531
2,Cross-encoder over top-10 only,0.0615
3,"Cross-encoder over 1,000,000 chunks (extrapola...",6146.7617


Cross-encoder cost ≈ 6.15 ms per (query, chunk) pair on this machine


## Step 12: Save results for the report

In [20]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

summary_df.to_csv(RESULTS_DIR / "failure_query_summary.csv", index=False)
latency_df.to_csv(RESULTS_DIR / "latency_per_query.csv", index=False)
latency_summary.to_csv(RESULTS_DIR / "latency_summary.csv", index=False)
scaling_df.to_csv(RESULTS_DIR / "scaling.csv", index=False)

with open(RESULTS_DIR / "rank_shifts_and_answers.json", "w", encoding="utf-8") as f:
    json.dump([
        {
            "category": r["category"], "query": r["query"], "gold": r["gold"], "gold_rule": r["gold_rule"],
            "rank_table": rank_shift_table(r, r["gold"]).drop(columns="preview").to_dict(orient="records"),
            "answer1_baseline": r["answer1"], "answer2_reranked": r["answer2"],
        }
        for r in all_results
    ], f, indent=2, ensure_ascii=False, default=str)

print("Saved:", sorted(p.name for p in RESULTS_DIR.iterdir()))

Saved: ['failure_query_summary.csv', 'latency_per_query.csv', 'latency_summary.csv', 'rank_shifts_and_answers.json', 'scaling.csv']
